# GGUF LLM/VLM Colab Backend Launcher
Tailored for **Google Colab T4 x1 GPU** runtimes. Serves local GGUF models over an OpenAI-compatible HTTP interface exposed publicly via Ngrok or Cloudflare Quick Tunnel fallbacks.

In [ ]:
# Cell 1: Clone/Install repository and import backend elements
import os
import sys

REPO_URL = "https://github.com/gutris1/llama-gguf-notebook-backend.git"
BRANCH = "main"
REPO_DIR = "/content/llama-gguf-notebook-backend"

if not os.path.exists(REPO_DIR):
    !git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}

sys.path.insert(0, REPO_DIR)
%cd {REPO_DIR}

from gguf_backend import diagnostics, installer, downloader, server, client, tunnel

In [ ]:
# Cell 2: Run environment diagnostics checks
diagnostics.run(profile="colab_t4x1")

In [ ]:
# Cell 3: Install prebuilt CUDA-enabled llama-server binaries and utilities
installer.install_runtime(profile="colab_t4x1")

In [ ]:
# Cell 4: Download model GGUF and optional projector
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF/resolve/main/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf"
MMPROJ_URL = "" # Optional mmproj GGUF URL (empty string for text-only)
HF_TOKEN = "" # Optional HuggingFace access token for private repos

downloader.download_model(
    model_url=MODEL_URL,
    mmproj_url=MMPROJ_URL,
    connections=16,
    hf_token=HF_TOKEN
)

In [ ]:
# Cell 5: Setup runtime config, launch inference server, and warmup
cfg = server.ServerConfig(
    profile="colab_t4x1",
    alias="local-vl",
    ctx_size=4096,
    flash_attn=True,
    cache_type_k="f16",
    cache_type_v="f16",
    image_min_tokens=None, # None = use model/mmproj defaults
    image_max_tokens=None,
    chat_template_kwargs=None # e.g. '{"enable_thinking":true}' to enable support
)

server.start_and_warmup(cfg)

In [ ]:
# Cell 6: Perform local chat completion API test
client.test_text(
    base_url="http://127.0.0.1:8080",
    model="local-vl",
    prompt="hallo, jawab satu kalimat pendek bahwa backend aktif."
)

In [ ]:
# Cell 7: Expose server via public tunnel and verify connectivity
# Look up from environment or assign manual auth token string
NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN", "")

PUBLIC_URL = tunnel.open_tunnel(
    port=8080,
    ngrok_token=NGROK_AUTHTOKEN,
    prefer="ngrok",
    fallback="cloudflare"
)

client.test_text(
    base_url=PUBLIC_URL,
    model="local-vl",
    prompt="hello dari public tunnel, jawab satu kalimat pendek."
)